# Reproducing the results of MacInnis et. al., 2026

This notebook will show you how to reproduce the results presented in [**TODO:LINK2PAPER**](https://arxiv.org/abs/XXXX.XXXXX) by applying the extragalactic foreground (FG) cleaning procedure described in that work to [ultrahigh-resolution simulations](https://github.com/CMB-HD/hdsims) of the lensed CMB, thermal and kinetic Sunyaev-Zel'dovich effect (tSZ and kSZ, respectively), the cosmic infrared background (CIB), radio galaxies, and instrumental noise at 90, 148, 219, and 277 GHz on a 100 square degree patch of the sky. 





The notebook is organized into different parts, summarized below:

**Part 1**: This is where you will provide any necessary paths to output directories, etc. (**TODO** : be more descriptive)

**Part 2**: Save the simulations used for FG cleaning 

**Part 3**: Run the FG cleaning procedure 
- Apply matched filters to the maps to iteratively detect, measure, and remove point sources (CIB and radio galaxies) and tSZ clusters from the maps at each frequency
  - The catalogs of the sources and clusters that were removed from the maps, as well as the maps themselves after FG cleaning, will be saved (along with additional, intermediate output files); we provide Python methods to load these files for you.
- Match the catalogs of detected sources and clusters to the true catalogs of all sources and clusters in the maps
- Take the power spectra of the FG-cleaned maps

**Part 4**: FG cleaning results
- Compare the power spectra of the FG-cleaned maps to the files provided with the [hdMockData](https://github.com/CMB-HD/hdMockData) repository
- Reproduce the plots in MacInnis et. al. of the FG-cleaned maps and power spectra, and the source and cluster recovery (Figures 1, 2, and 6 - 11)
- (**TODO**) Optionally, reproduce the parameter results in MacInnis et. al. (Tables 4 - 6 and Figures 12, 13)
  - **Note**: For this step, you will also need to install [hdfisher](https://github.com/CMB-HD/hdfisher) and [getdist](https://getdist.readthedocs.io) (in addition to the requirements of `hdfgclean`).

# Brief summary of the FG cleaning procedure

Below, we go into a little more detail about the FG cleaning procedure (see [**TODO:LINK2PAPER**](https://arxiv.org/abs/XXXX.XXXXX) for all details), and in particular what will happen when you run `hdfgclean`.

- We start with four maps (one at each of the at four frequencies 90, 148, 219, and 277 GHz); each map is the sum of the lensed CMB, tSZ, kSZ, CIB, and radio maps, each convolved with the pixel window function and CMB-HD instrumental beam, and the CMB-HD instrumental noise.
  - The maps we will use were generated with [hdsims](https://github.com/CMB-HD/hdsims), and they are available on [LAMBDA](https://lambda.gsfc.nasa.gov/simulation/ultrahigh_resolution_sims.html); we provide instructions on how to download them in "part 2".
  - The maps are $11^\circ \times 11^\circ$; they are apodized over a $0.5^\circ$ border at the map edges, leaving an inner $10^\circ \times 10^\circ$ un-apodized region; this inner region is used to calculate the results in [**TODO:LINK2PAPER**](https://arxiv.org/abs/XXXX.XXXXX).



(**TODO** : summarizing fg cleaning below) 


- the inner $10^\circ \times 10^\circ$ region of each map is divided into a grid of 25 non-overlapping smaller $2^\circ \times 2^\circ$ patches 
  - we remove FGs and do the matching between measured and true catalogs on each smaller patch (simultaneously, with MPI)
  - then we combine the detected source and cluster catalogs from all patches into a set of catalogs for the full map, use these to save the (full-sized) FG-cleaned maps, and then take the power spectrum of the inner $10^\circ \times 10^\circ$ region of these maps at 90 and 148 GHz


- we remove FGs by iteratively detecting, measuring, and subtracting point sources (CIB+radio) and (tSZ) clusters from the maps, first removing objects detected above a high signal-to-noise ratio (SNR) threshold, and then, once there are no more detections above that SNR, moving down to lower SNR thresholds (until a minimum SNR = 4) 
   - on each iteration, we detect and measure sources/clusters by applying a filter, matched to the beam profile for point sources (CIB+radio) or a set of cluster profiles for the tSZ, to the map(s) (which, after the first iteration, will be partially FG-cleaned) 
     - for point sources, this is done at each frequency individually ; for clusters, we apply a multi-frequency matched filter to the four maps simultaneously
     - we remove point sources at each frequency first ; then, we remove clusters from the maps


- Note that we do not remove the kSZ signal from the maps
  - when we say "FG cleaning", we mean removing tSZ, CIB, radio from maps
  - but when we refer to the "FGs" in a map, we mean kSZ + tSZ + CIB + radio


- to calculate the matched filters, we need an estimate of the noise in the map being filtered; in this case, "noise" refers to anything in the map that isn't the signal we are trying to measure (e.g., when we are filtering the maps to find CIB and radio sources, the "noise" is tSZ + kSZ + CMB + instrumental noise)
  - we use a second realization of the simulations for this purpose; we will refer to these as "noise maps" 
  - **TODO** : fill in the details here
    - **the point is**: you will need to generate a second set of $3^\circ \times 3^\circ$ hdsims (they're 3x3 because the filters are applied to the smaller patches)  before running the fg cleaning on the larger maps ; we provide instructions on how to do that below

---

# Part 1 : 

Below, we import the packages required to run this notebook; from the `hdfgclean` package, we import two modules:
- `hdfgclean.py` contains the main `HDFGClean` class, used to run the FG cleaning and load in the resulting maps and catalogs
- `hdfgclean_utils.py` contains functions that are only used in this notebook to print out instructions, and compare the power spectra of the FG-cleaned maps to the files provided with the [hdMockData](https://github.com/CMB-HD/hdMockData) repository

In [ ]:
import os 
from hdfgclean import hdfgclean, hdfgclean_utils

In the cell below, you **must** provide the paths to:
- your `hd_sims_dir` where the simulations will be saved (see [hdsims](https://github.com/CMB-HD/hdsims) for more information)
- an `output_dir` where the output from FG cleaning will be saved

**Note** that you will need about 80 GB of space to save the simulations, and about 45 GB to save the FG cleaning output

In [ ]:
hd_sims_dir = 
output_dir = 

(**TODO**) If you would also like to reproduce the parameter results of [**TODO:LINK2PAPER**](https://arxiv.org/abs/XXXX.XXXXX), set `reproduce_param_results=True` below.
- If you would also like to calculate the Fisher matrices yourself, set `calculate_fisher = True` below; otherwise, if `calculate_fisher = True`, we will load in pre-computed Fisher matrices

In [ ]:
#reproduce_param_results = False
#calculate_fisher = False

If you have moved (or made a copy of) this notebook into a different directory, provide the path to the `hdfgclean` repository (i.e., the directory that contains the readme file) below; otherwise, you can leave this cell unchanged:

In [ ]:
hdfgclean_repo_dir = None

**It is strongly recommended to use MPI** when running the FG cleaning (part 3). 

Below, you may change `num_mpi_processes_fgclean` (default is `25`) and `num_mpi_processes_spectra` (default is `2`), the number of MPI processes to use for the FG cleaning on the smaller patches or the power spectra of the full maps, respectively. If you're not using MPI, set each to `1`; however, in this case, we recommend running FG cleaning on a single smaller patch with the `example_2x2.ipynb` notebook we provide (**TODO**)

- The input maps will be divided into 25 smaller patches, and the FG cleaning procedure can be run on each patch simultaneously, so we recommend running the FG cleaning with 25 MPI processes (one per patch), *and* running on multiple compute nodes.
  - While testing on the Stony Brook University [SeaWulf](https://rci.stonybrook.edu/HPC/understanding-SeaWulf) cluster, we used five nodes in the `long-96core` [queue](https://rci.stonybrook.edu/HPC/faqs/seawulf-queues) to FG clean the maps, which took about five hours.
  - (Note that, without MPI, the same FG cleaning run would have take about 5 hours $\times$ 25 patches = 125 hours total!)
- After FG cleaning the maps, we will match the catalogs of detected sources and clusters to the catalogs of true sources and clusters in the maps; this can also be done simultaneously on each patch.
  - This step requires less memory, so we ran it on a single node in the `short-96core` queue, which took about 1.5 hours.
- Then we take the power spectra of the full 100 square degree maps; this requires much more memory, so we recommend using fewer MPI processes for this step.
  - We used two MPI processes on a single node in the `hbm-short-96core` queue, which took about an hour.
  
If you are running the FG cleaning by submitting jobs on a cluster using [slurm](https://slurm.schedmd.com) (e.g. with the [sbatch](https://slurm.schedmd.com/sbatch.html) command), you can use the `--dependency` option to ensure that the first step (FG cleaning the maps) is completed before trying to match catalogs or take power spectra; see [here](https://hpc.nih.gov/docs/job_dependencies.html) for examples

In [ ]:
num_mpi_processes_fgclean = 25
num_mpi_processes_spectra = 2

---

# Part 2: Save the simulations

In the following two cells, we will print out intructions to download the HD simulations from LAMBDA, and to generate the smaller set of "noise maps"

(**TODO**) to generate the noise maps, you do *not* need to download the full-sky S10 simulations ; we provide the necessary lower-resolution S10 sims for the small patch of sky on github

In [ ]:
hdfgclean_utils.print_instructions_to_download_hdsims(hd_sims_dir, hdfgclean_repo_dir=hdfgclean_repo_dir)

In [ ]:
hdfgclean_utils.print_instructions_to_generate_noise_sims(hd_sims_dir, hdfgclean_repo_dir=hdfgclean_repo_dir, output_dir=output_dir)

After following the instructions above, run the cell below to make sure the files were saved correctly; if they weren't, follow the instructions that are printed out:

In [ ]:
# check if the simulation files have been saved:
hdsims_are_saved = hdfgclean_utils.hdsims_files_are_saved(hd_sims_dir)
maps_for_noise_are_saved = hdfgclean_utils.noise_map_files_are_saved(hd_sims_dir)
sim_files_saved = hdsims_are_saved and maps_for_noise_are_saved
if sim_files_saved:
    print("All necessary maps for the matched filter calculations have been saved.")
else:
    # print out the instructions again:
    if not hdsims_are_saved:
        hdfgclean_utils.print_instructions_to_download_hdsims(hd_sims_dir, hdfgclean_repo_dir=hdfgclean_repo_dir)
    if not maps_for_noise_are_saved:
        hdfgclean_utils.print_instructions_to_generate_noise_sims(hd_sims_dir, output_dir=output_dir, hdfgclean_repo_dir=hdfgclean_repo_dir)
    print(f"\nThen, re-run this cell.")

---

# Part 3 : Running FG cleaning

**TODO** : explain:
- the `HDFGClean` class in the `hdfgclean.py` module is the "main" class: use it to run FG cleaning (the `run_hdfgclean` method, which is what `reproduce_10x10.py` calls), and to get results (FG cleaned maps, catalogs, etc.)
- there are lots of different settings that can be changed for the sims (same as `hdsims`, e.g. patch size, location) or for FG cleaning (defined in the `HDFGCleanMaps` class of `hdfgclean_maps.py` module), *but* everything has been tested with the default settings
  - minimum requirements are the `hd_sims_dir` and `output_dir`
- you can pass the settings directly when initializing the class, or you can save them in a config `.yaml` file, and just pass the file name to the class ; that is what we do below

Below, we will save a `.yaml` file with your `hd_sims_dir` and `output_dir`; this will be used to initialize the `HDFGClean` class (in this notebook and in  `reproduce_10x10.py`)

Then, we will print out instructions to run the FG cleaning, using the `reproduce_10x10.py` python script.

In [ ]:
# save a configuration file to run the FG cleaning using `reproduce_10x10.py`
config_file = os.path.join(output_dir, 'hdfgclean.yaml')    
if not os.path.exists(config_file):
    hdfgclean.HDFGClean.save_config(config_file, hd_sims_dir, output_dir=output_dir)

In [ ]:
hdfgclean_utils.print_instructions_to_reproduce_10x10(config_file, hdfgclean_repo_dir=hdfgclean_repo_dir, 
                                                      num_mpi_processes_fgclean=num_mpi_processes_fgclean, 
                                                      num_mpi_processes_spectra=num_mpi_processes_spectra)

---

# Part 4: FG cleaning results

Below, we initialize the `HDFGClean` class. 

Note that, if this is the first time initializing `HDFGClean` with this `config_file`, the maps that are input to the FG cleaning procedure will be loaded in and saved: these maps are the beam-convolved sum of the lensed CMB, kSZ, tSZ, CIB, and radio maps, plus the instrumental noise maps; each map is about 2 GB. (However, you shouldn't have to worry about this if you've followed the instructions above!)

In [ ]:
fgcleanlib = hdfgclean.HDFGClean.from_config(config_file)

Note that, alternatively, we could have also initialized `HDFGClean` with:

```
fgcleanlib = hdfgclean.HDFGClean(output_dir, hd_sims_dir)
```

The `HDFGClean` class also accepts additional keyword arguments; for more information, refer to the documentation for the `HDFGCleanMaps` class in the `hdfgclean_maps` module (because the `HDFGClean` class inherits its initialization method from `HDFGCleanMaps`).

Now we will check if everything has been saved:

In [ ]:
fgclean_files_saved = hdfgclean_utils.all_fgclean_files_are_saved(fgcleanlib, config_file, 
                                                                  num_mpi_processes_fgclean=num_mpi_processes_fgclean, 
                                                                  num_mpi_processes_spectra=num_mpi_processes_spectra)

Below, we compare the power spectra of your 90 and 148 GHz FG-cleaned maps to the files provided with hdMockData:

(**TODO** : explain what each is)

In [ ]:
if fgclean_files_saved:
    hdfgclean_utils.compare_10x10_spectra(fgcleanlib)


---


In the following cells, we will plot Figures 1, 2, and 6 - 11 of MacInnis et. al. 

(**TODO** : say a bit more / explain what is being plotted and how to get the spectra, etc. used in plots )

## plot power spectra

In [ ]:
if fgclean_files_saved:
    fgcleanlib.plot_coadded_fgcleaned_spectra()

In [ ]:
if fgclean_files_saved:
    fgcleanlib.plot_fgcleaned_spectra()

## plots for source and cluster recovery

In [ ]:
if fgclean_files_saved:
    fgcleanlib.flux_measurement_plot()

In [ ]:
if fgclean_files_saved:
    fgcleanlib.sources_plot()

In [ ]:
if fgclean_files_saved:
    fgcleanlib.cluster_mass_vs_redshift_plot()

In [ ]:
if fgclean_files_saved:
    fgcleanlib.cluster_completeness_per_mass_redshift_plot()

## plots of maps before and after FG cleaning

In [ ]:
if fgclean_files_saved:
    fgcleanlib.plot_fgcleaned_maps_with_cmb()

In [ ]:
if fgclean_files_saved:
    fgcleanlib.plot_fgcleaned_maps_without_cmb()

---

## (TODO) parameter forecasts

In [ ]:
if reproduce_param_results:
    from hdfisher import utils as hdutils, fisher, dataconfig, config
    #from getdist import plots as ps
    #from getdist.gaussian_mixtures import GaussianND
